# Twelve Data → Bronze Market Prices Ingestion

Ingests daily end-of-day equity prices for ~100 symbols from the **Twelve Data** HTTP API into the Delta bronze table `workspace.bronze.market_prices_daily`. The pipeline pulls data via Python `requests`, stages it in pandas, converts to a PySpark DataFrame with an explicit schema, applies basic validation (null checks, positive close-price filter, duplicate removal), and performs a Delta **MERGE** on the business key `(ticker, trade_date)` for idempotent writes.

**Modes:** `BACKFILL` pulls a large historical window (5,000 bars per symbol); `INCREMENTAL` pulls the most recent 5 bars. Run BACKFILL once, then schedule INCREMENTAL daily.

**Key config:** API key from Databricks secret scope `market-data`, 8-second rate-limit delay between requests, 3 retries with exponential backoff.

In [0]:
# Databricks notebook source
# ============================================================
# TWELVE DATA -> BRONZE MARKET PRICES
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import time
import requests
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    current_timestamp,
    lit
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    DateType
)

from delta.tables import DeltaTable


spark = SparkSession.builder.getOrCreate()


# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_URL = "https://api.twelvedata.com/time_series"

TARGET_TABLE = "bronze.market_prices_daily"

DATA_SOURCE = "Twelve Data"

INTERVAL = "1day"


# ------------------------------------------------------------
# LOAD MODE
# ------------------------------------------------------------
#
# BACKFILL:
#   Large historical pull.
#
# INCREMENTAL:
#   Pull only recent observations.
#

LOAD_MODE = dbutils.widgets.get("load_mode")


if LOAD_MODE == "BACKFILL":

    OUTPUT_SIZE = 5000

elif LOAD_MODE == "INCREMENTAL":

    # Pull several days rather than exactly one.
    #
    # This allows the job to recover automatically if:
    #
    # - yesterday's job failed
    # - there was a holiday
    # - the pipeline missed a trading day
    #
    # Delta MERGE prevents duplicates.

    OUTPUT_SIZE = 5

else:

    raise ValueError(
        "LOAD_MODE must be BACKFILL or INCREMENTAL"
    )


# ------------------------------------------------------------
# UNIVERSE SETTINGS
# ------------------------------------------------------------

MAX_SYMBOLS = 100


# ------------------------------------------------------------
# RATE LIMITING
# ------------------------------------------------------------
#
# A conservative delay between requests.
#
# Adjust this if your Twelve Data plan changes.

REQUEST_DELAY_SECONDS = 8


# ------------------------------------------------------------
# RETRY SETTINGS
# ------------------------------------------------------------

MAX_RETRIES = 3

RETRY_DELAY_SECONDS = 15


# ============================================================
# 3. GET API KEY FROM DATABRICKS SECRET
# ============================================================

API_KEY = dbutils.secrets.get(
    scope="market-data",
    key="twelve-data-api-key"
)


# Defensive cleanup.
#
# You discovered that the stored secret currently contains
# a NULL byte. Ideally re-save the Databricks secret correctly
# and eventually remove this line.

API_KEY = API_KEY.replace("\x00", "").strip()


if not API_KEY:

    raise RuntimeError(
        "Twelve Data API key is empty."
    )


print("Twelve Data API key loaded successfully.")


# ============================================================
# 4. SYMBOL UNIVERSE
# ============================================================
#
# OPTION A:
# Use a manually defined list.
#
# Replace or expand this list as desired.
#
# Eventually I recommend sourcing this from your
# Silver SEC company table instead.
# ============================================================


SYMBOLS = [

    "AAPL",
    "SPY",
    "MSFT",
    "NVDA",
    "AMZN",
    "GOOGL",
    "META",
    "BRK.B",
    "AVGO",
    "TSLA",
    "JPM",

    "LLY",
    "V",
    "WMT",
    "MA",
    "XOM",
    "UNH",
    "ORCL",
    "COST",
    "HD",
    "PG",

    "JNJ",
    "NFLX",
    "ABBV",
    "BAC",
    "KO",
    "CRM",
    "CVX",
    "MRK",
    "AMD",
    "PEP",

    "TMO",
    "WFC",
    "CSCO",
    "ACN",
    "LIN",
    "MCD",
    "ABT",
    "IBM",
    "GE",
    "PM",

    "AXP",
    "ISRG",
    "CAT",
    "QCOM",
    "VZ",
    "INTU",
    "TXN",
    "NOW",
    "DIS",
    "AMGN",

    "NEE",
    "RTX",
    "SPGI",
    "PFE",
    "LOW",
    "GS",
    "BKNG",
    "HON",
    "BLK",
    "UNP",

    "T",
    "SYK",
    "ELV",
    "PLD",
    "VRTX",
    "ADP",
    "MDT",
    "TJX",
    "MMC",
    "ADI",

    "CB",
    "DE",
    "LMT",
    "BMY",
    "GILD",
    "AMT",
    "C",
    "SCHW",
    "ETN",
    "CI",

    "SO",
    "MO",
    "PANW",
    "FI",
    "COP",
    "DUK",
    "ICE",
    "SHW",
    "UPS",
    "CL",

    "ZTS",
    "MCK",
    "USB",
    "BDX",
    "WM",
    "EOG",
    "AON",
    "APD",
    "ITW",
    "PNC"

]


# Normalize symbols

SYMBOLS = [

    symbol.strip().upper()

    for symbol in SYMBOLS

    if symbol.strip()

]


# Remove duplicates while preserving order

SYMBOLS = list(dict.fromkeys(SYMBOLS))


# Limit daily universe

SYMBOLS = SYMBOLS[:MAX_SYMBOLS]


print(
    f"Symbols scheduled for ingestion: "
    f"{len(SYMBOLS)}"
)


# ============================================================
# 5. OPTIONAL: USE SEC COMPANY TABLE INSTEAD
# ============================================================
#
# Eventually you can replace the manual list above with:
#
# SYMBOLS = [
#     row["ticker"]
#     for row in (
#         spark.table("silver.sec_companies")
#         .select("ticker")
#         .where(col("ticker").isNotNull())
#         .distinct()
#         .orderBy("ticker")
#         .limit(MAX_SYMBOLS)
#         .collect()
#     )
# ]
#
# This connects:
#
# SEC company universe
#       ↓
# Twelve Data
#       ↓
# market-price history
#
# ============================================================


# ============================================================
# 6. API REQUEST FUNCTION
# ============================================================

def fetch_symbol(symbol):

    """
    Fetch daily market prices for a single symbol.

    Returns:
        Pandas DataFrame on success.
        None on failure.
    """

    params = {

        "apikey": API_KEY,

        "symbol": symbol,

        "interval": INTERVAL,

        "outputsize": OUTPUT_SIZE,

        "format": "JSON"

    }


    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            response = requests.get(

                BASE_URL,

                params=params,

                timeout=60

            )


            # ------------------------------------------------
            # HANDLE RATE LIMITING
            # ------------------------------------------------

            if response.status_code == 429:

                print(
                    f"{symbol}: Rate limited. "
                    f"Retrying..."
                )

                time.sleep(
                    RETRY_DELAY_SECONDS * attempt
                )

                continue


            # ------------------------------------------------
            # HANDLE SERVER ERRORS
            # ------------------------------------------------

            if response.status_code >= 500:

                print(
                    f"{symbol}: Server error "
                    f"{response.status_code}. "
                    f"Retrying..."
                )

                time.sleep(
                    RETRY_DELAY_SECONDS * attempt
                )

                continue


            response.raise_for_status()


            data = response.json()


            # ------------------------------------------------
            # TWELVE DATA APPLICATION ERROR
            # ------------------------------------------------

            if data.get("status") == "error":

                print(
                    f"{symbol}: API error: "
                    f"{data.get('message')}"
                )

                return None


            # ------------------------------------------------
            # NO PRICE DATA
            # ------------------------------------------------

            if "values" not in data:

                print(
                    f"{symbol}: No values returned."
                )

                return None


            # ------------------------------------------------
            # CONVERT TO DATAFRAME
            # ------------------------------------------------

            pdf = pd.DataFrame(
                data["values"]
            )


            if pdf.empty:

                print(
                    f"{symbol}: Empty dataset."
                )

                return None


            # ------------------------------------------------
            # ADD SYMBOL
            # ------------------------------------------------

            pdf["ticker"] = symbol


            # ------------------------------------------------
            # CONVERT DATE
            # ------------------------------------------------

            pdf["trade_date"] = pd.to_datetime(
                pdf["datetime"],
                errors="coerce"
            ).dt.date


            # ------------------------------------------------
            # NUMERIC CONVERSIONS
            # ------------------------------------------------

            for column in [

                "open",
                "high",
                "low",
                "close"

            ]:

                pdf[column] = pd.to_numeric(

                    pdf[column],

                    errors="coerce"

                )


            pdf["volume"] = pd.to_numeric(

                pdf["volume"],

                errors="coerce"

            )


            # ------------------------------------------------
            # STANDARDIZE COLUMNS
            # ------------------------------------------------

            pdf = pdf.rename(
                columns={

                    "open":
                        "open_price",

                    "high":
                        "high_price",

                    "low":
                        "low_price",

                    "close":
                        "close_price"

                }
            )


            # ------------------------------------------------
            # ADD SOURCE METADATA
            # ------------------------------------------------

            pdf["data_source"] = DATA_SOURCE


            # ------------------------------------------------
            # SELECT FINAL BRONZE FIELDS
            # ------------------------------------------------

            pdf = pdf[[

                "ticker",

                "trade_date",

                "open_price",

                "high_price",

                "low_price",

                "close_price",

                "volume",

                "data_source"

            ]]


            print(
                f"{symbol}: "
                f"{len(pdf):,} observations downloaded."
            )


            return pdf


        except requests.exceptions.RequestException as e:

            print(
                f"{symbol}: Request failed "
                f"(attempt {attempt}/{MAX_RETRIES})"
            )

            print(str(e))


            if attempt < MAX_RETRIES:

                time.sleep(
                    RETRY_DELAY_SECONDS * attempt
                )


    print(
        f"{symbol}: Failed after "
        f"{MAX_RETRIES} attempts."
    )

    return None


# ============================================================
# 7. INGEST SYMBOLS
# ============================================================

frames = []

successful_symbols = []

failed_symbols = []


for index, symbol in enumerate(
    SYMBOLS,
    start=1
):

    print(
        f"\n[{index}/{len(SYMBOLS)}] "
        f"Downloading {symbol}"
    )


    pdf = fetch_symbol(symbol)


    if pdf is not None:

        frames.append(pdf)

        successful_symbols.append(symbol)

    else:

        failed_symbols.append(symbol)


    # Don't sleep after final request

    if index < len(SYMBOLS):

        time.sleep(
            REQUEST_DELAY_SECONDS
        )


# ============================================================
# 8. CHECK RESULTS
# ============================================================

if not frames:

    raise RuntimeError(
        "No market-price data was downloaded."
    )


if failed_symbols:

    print(
        "Failed symbols:",
        failed_symbols
    )


# ============================================================
# 9. COMBINE DATA
# ============================================================

combined_pdf = pd.concat(

    frames,

    ignore_index=True

)


print(
    f"Total API observations downloaded: "
    f"{len(combined_pdf):,}"
)


# ============================================================
# 10. BASIC PANDAS VALIDATION
# ============================================================

combined_pdf = combined_pdf.drop_duplicates(
    subset=[
        "ticker",
        "trade_date"
    ],
    keep="last"
)


print(
    f"Valid observations after "
    f"deduplication: "
    f"{len(combined_pdf):,}"
)


# ============================================================
# 11. DEFINE SPARK SCHEMA
# ============================================================

market_schema = StructType([

    StructField(
        "ticker",
        StringType(),
        False
    ),

    StructField(
        "trade_date",
        DateType(),
        False
    ),

    StructField(
        "open_price",
        DoubleType(),
        True
    ),

    StructField(
        "high_price",
        DoubleType(),
        True
    ),

    StructField(
        "low_price",
        DoubleType(),
        True
    ),

    StructField(
        "close_price",
        DoubleType(),
        True
    ),

    StructField(
        "volume",
        DoubleType(),
        True
    ),

    StructField(
        "data_source",
        StringType(),
        False
    )

])


# ============================================================
# 12. CREATE SPARK DATAFRAME
# ============================================================

spark_df = spark.createDataFrame(

    combined_pdf,

    schema=market_schema

)


bronze_df = (

    spark_df

    .withColumn(
        "volume",
        col("volume").cast("long")
    )

    .withColumn(
        "adjustment_type",
        lit("split_dividend_adjusted")
    )

    .withColumn(
        "source_file",
        lit("Twelve Data API")
    )

    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )

)


# ============================================================
# 13. BASIC DATA QUALITY FILTER
# ============================================================

bronze_df = bronze_df.filter(

    col("ticker").isNotNull()

    & col("trade_date").isNotNull()

    & col("close_price").isNotNull()

    & (col("close_price") > 0)

)


# ============================================================
# 14. CREATE BRONZE SCHEMA
# ============================================================

spark.sql("""

CREATE SCHEMA IF NOT EXISTS bronze

""")


# ============================================================
# 17. CREATE TARGET TABLE
# ============================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (

    ticker STRING,

    trade_date DATE,

    open_price DOUBLE,

    high_price DOUBLE,

    low_price DOUBLE,

    close_price DOUBLE,

    volume BIGINT,

    adjustment_type STRING,

    data_source STRING,

    source_file STRING,

    ingestion_timestamp TIMESTAMP

)

USING DELTA

COMMENT
'Daily end-of-day equity market prices from Twelve Data'

""")


# ============================================================
# 18. DELTA MERGE
# ============================================================
#
# Primary business key:
#
#     ticker + trade_date
#
# This makes the pipeline idempotent.
#
# Re-running the job does NOT create duplicate daily prices.
#
# ============================================================

target = DeltaTable.forName(

    spark,

    TARGET_TABLE

)


(

    target.alias("target")

    .merge(

        bronze_df.alias("source"),

        """

        target.ticker = source.ticker

        AND

        target.trade_date = source.trade_date

        AND

        target.adjustment_type = source.adjustment_type

        """

    )

    .whenMatchedUpdate(

        set={

            "open_price":
                "source.open_price",

            "high_price":
                "source.high_price",

            "low_price":
                "source.low_price",

            "close_price":
                "source.close_price",

            "volume":
                "source.volume",

            "data_source":
                "source.data_source",

            "ingestion_timestamp":
                "source.ingestion_timestamp"

        }

    )

    .whenNotMatchedInsertAll()

    .execute()

)


print(
    "\nDelta MERGE completed successfully."
)


# ============================================================
# 19. VALIDATE CURRENT INGESTION
# ============================================================

validation_df = (

    spark.table(TARGET_TABLE)

    .filter(
        col("ticker").isin(
            successful_symbols
        )
    )

    .groupBy("ticker")

    .agg(

        {
            "trade_date": "count"
        }

    )

    .orderBy("ticker")

)


display(validation_df)


# ============================================================
# 20. TABLE SUMMARY
# ============================================================

summary_df = spark.sql(f"""

SELECT

    COUNT(*) AS total_records,

    COUNT(DISTINCT ticker)
        AS total_symbols,

    MIN(trade_date)
        AS earliest_trade_date,

    MAX(trade_date)
        AS latest_trade_date,

    MAX(ingestion_timestamp)
        AS latest_ingestion

FROM {TARGET_TABLE}

""")


display(summary_df)


# ============================================================
# 21. MOST RECENT OBSERVATIONS
# ============================================================

latest_df = spark.sql(f"""

SELECT

    ticker,

    trade_date,

    open_price,

    high_price,

    low_price,

    close_price,

    volume,

    data_source,

    ingestion_timestamp

FROM {TARGET_TABLE}

ORDER BY

    trade_date DESC,

    ticker

LIMIT 100

""")


display(latest_df)


# ============================================================
# 22. JOB SUMMARY
# ============================================================

print("\n========================================")
print("TWELVE DATA INGESTION COMPLETE")
print("========================================")

print(
    f"Load mode: {LOAD_MODE}"
)

print(
    f"Requested symbols: {len(SYMBOLS)}"
)

print(
    f"Successful symbols: "
    f"{len(successful_symbols)}"
)

print(
    f"Failed symbols: "
    f"{len(failed_symbols)}"
)

print(
    f"API observations received: "
    f"{len(combined_pdf):,}"
)

print(
    f"Target table: {TARGET_TABLE}"
)

print("========================================")